# Data Preprocessing and Converting Excel to Parquet

In [ ]:
# Packages
import numpy as np
import pandas as pd
import os
import re
from tqdm import tqdm

# Show working directory
print(os.getcwd())

# Retrieve unprocessed data if it is stored
%store -r static_data_unprocessed
%store -r restaurant_data_unprocessed

# Import data if it isn't stored
if 'static_data_unprocessed' not in locals():

    # Initialize a dictionary for the four Excel files of "static reference data"
    static_data_separated = {}
    for batch_index in [1,2]:

        # Retrieve filnames
        static_data_filenames = os.listdir(f"data/0_data_excel/batch_{batch_index}")
        for filename in tqdm(static_data_filenames, leave=False):

            # Exclude other folders
            if ".xlsx" in filename:
                df = pd.read_excel(f"data/0_data_excel/batch_{batch_index}/{filename}")
                base_name = re.sub(r'\.xlsx$', '', filename)
                static_data_separated[f"{base_name}_{batch_index}"] = df

    # Rename and store
    static_data_unprocessed = static_data_separated.copy()
    %store static_data_unprocessed

# Data already exists
else:
    static_data_separated = static_data_unprocessed.copy()

# Check if the data is already imported
if 'restaurant_data_unprocessed' not in locals():

    # We have two data containers, one for dataframes directly to be added, and another for two parts of dataframes to be merged then added

    # Initialize a dictionary for the 30 Excel files of "restaurant sales data"
    restaurant_data = {}
    partial_data_filenames = []
    for batch_index in [1,2]:

        # Retrieve filnames
        location_filenames = os.listdir(f"data/0_data_excel/batch_{batch_index}/orders_item_level")
        for location_filename in tqdm(location_filenames):
            
            # Keep a version without the file extension
            location_id_with_part = re.sub(r'\.xlsx$', '', location_filename)
            
            # Keep a version without the file extension or the part addition
            location_id = re.sub(r'(_part[12])?\.xlsx$', '', location_filename)

            # To be directly added: check if it is a single file do not add it to be merged
            if "part" not in location_filename:
                df = pd.read_excel(f"data/0_data_excel/batch_{batch_index}/orders_item_level/" + location_filename)
                restaurant_data[location_id_with_part] = df

            # To be merged: if it is a partial dataframe, then select the first part
            elif "part1" in location_filename:
                partial_data_filenames.append(location_id)

            # else part 2, then that will be caught on the next loop

    # Read dataframes with a part1 and part2 then merge
    for location_id in tqdm(partial_data_filenames):

            # Add both parts to a list to concat
            to_concat = []
            for part in [1,2]:
                df = pd.read_excel(f"data/0_data_excel/batch_1/orders_item_level/{location_id}_part{part}.xlsx")
                to_concat.append(df)
            restaurant_data[location_id] = pd.concat(to_concat)

    # Rename and store
    restaurant_data_unprocessed = {}
    for loc_id, df in restaurant_data.items():
        restaurant_data_unprocessed[loc_id] = df.copy()
    %store restaurant_data_unprocessed

# Data already exists
else:
    restaurant_data = {}
    for loc_id, df in restaurant_data_unprocessed.items():
        restaurant_data[loc_id] = df.copy()
     


# Separate data parts 1 and 2 to merge

# Separate out items_tagged, so that v1 and v2 can be crosschecked
items_tagged_1 = static_data_separated['items_tagged_1']
items_tagged_v2_1 = static_data_separated['items_tagged_v2_1']
items_tagged_2 = static_data_separated['items_tagged_2']

# Create a new dict
static_data_without_items_tagged = static_data_separated.copy()

# Remove the items_tagged data from the new dict
if 'items_tagged_1' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_1']
if 'items_tagged_v2_1' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_v2_1']
if 'items_tagged_2' in static_data_without_items_tagged:
    del static_data_without_items_tagged['items_tagged_2']

# Initialize lists for the two parts
static_data_batch_1 = []
static_data_batch_2 = []
for name, df in static_data_without_items_tagged.items():

    # Add batch 1 numbering
    if "1" in name:
        df.columns.name = name
        df['batch'] = 1
        static_data_batch_1.append(df)

    # Add batch 2 numbering
    else:
        df.columns.name = name
        df['batch'] = 2
        static_data_batch_2.append(df)
        
        
    
# Combine data parts 1 and 2   
        
# Reinitialize dict for static data
static_data = {}

# Standardize number of columns: version two has an item description while version 1 does not
items_tagged_v2_1['item_description'] = np.nan
items_tagged_v2_1 = items_tagged_v2_1[items_tagged_2.columns.tolist()]

# Merge
items_tagged = pd.concat([items_tagged_v2_1, items_tagged_2])
items_tagged.reset_index(drop=True, inplace=True)

# First add the items_tagged menu data
static_data['items_tagged'] = items_tagged.copy()

# Merge the rest and add
for df1, df2 in zip(static_data_batch_1, static_data_batch_2):

    # Check number of rows in the two parts
    print(df1.columns.name, df1.shape)
    print(df2.columns.name, df2.shape)

    # Concat
    df = pd.concat([df1, df2])
    df.reset_index(drop=True, inplace=True)

    # Remove the part for the final name
    df.columns.name = re.sub(r'_[12]', '', df1.columns.name)
    
    static_data[df.columns.name] = df.copy()
    
    

# Timezones

# Determine timezones
timezones_acronyms = {}
for loc_id, df in restaurant_data.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
    
# Timezone mapping
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}


# Function to handle the conversion to string while keeping NaNs
def to_string_or_na(series):
    return series.where(series.isna(), series.astype(str))


### Static reference data

# Format menu data
static_data['items_tagged'] = (static_data['items_tagged']
                               .astype({'is_plant_based':str}) # Convert types before using accessors (str, dt) to prevent data loss
                               .assign(item_name = lambda df: df['item_name'].pipe(to_string_or_na), # Keep NaNs
                                       is_plant_based = lambda df: (df['is_plant_based']
                                                                    .pipe(title_keep_ids) # Capitalize
                                                                    .str.replace('.', '') # Clean string artifacts
                                                                    .astype('category')))) # Make category (since values won't be modified)

# Format location data
static_data['locations'] = (static_data['locations']
                            .assign(zip_code = lambda df: df['zip_code'].pipe(to_string_or_na)) # Keep NaNs
                            .set_index(keys='location_id', drop=False)) # Set indices    

# Format promo data
static_data['before_after_details'] = (static_data['before_after_details']
                                       .astype({'first_plant_based_mention':str})
                                       .assign(cross_over_date = lambda df: (df['cross_over_date']
                                                                             .pipe(pd.to_datetime)), # To datetime
                                               first_plant_based_mention = lambda df: df['first_plant_based_mention'].pipe(title_keep_ids)) # Capitalize
                                       .set_index(keys='location_id', drop=False))


### Restaurant sales data

# Format sales data
for loc_id, df in tqdm(restaurant_data.items()):
        
        static_data['before_after_details'].loc[loc_id,'cross_over_date'] = static_data['before_after_details'].loc[loc_id,'cross_over_date'].tz_localize(timezones[loc_id])

        # Fix encoding issues for parquet, keeping NaNs
        restaurant_data[loc_id] = (restaurant_data[loc_id]
                                   .astype({'unique_id':'category', 
                                            'order_id':'category',
                                            'location_id':'category',
                                            'customer_id':'category'})
                                   .assign(item_modifications = lambda df: df['item_modifications'].pipe(to_string_or_na), # Keep NaNs
                                           item_name = lambda df: df['item_name'].pipe(to_string_or_na), # Keep NaNs
                                           created_at = lambda df: (df['created_at']
                                                                    .str.replace(r'\s[A-Z]{3}', '', regex=True) # Remove duplicate timezone info
                                                                    .pipe(pd.to_datetime, utc=True) # Convert to datetime
                                                                    .dt.tz_convert(timezones[loc_id])),
                                           unit_price = lambda df: df['item_price'] / df['item_quantity']) # Add a unit price column
                                   .set_index(keys="created_at", drop=False) # Make time series and remove the alphabetical timezone identifier (there is a numerical one already)
                                   .sort_index())
        
        
before_after_details = static_data['before_after_details']
%store before_after_details

# Note: (most commpressed) gzip > zstd > snappy > lz4 (fastest compression)

# Static data
for name, df in tqdm(static_data.items()):

    # Write to parquet
    df.to_parquet(f"data/1_data_parquet/{name}.parquet", compression='zstd', index=True)

# Sales data
for loc_id, df in tqdm(restaurant_data.items()):

    # Write to parquet
    df.to_parquet(f"data/1_data_parquet/orders_item_level/{loc_id}.parquet", compression='zstd', index=True)

NA Checking

In [ ]:
total_entries = 0
total_na = 0
total_duplicated = 0
for loc_id, df in restaurant_data_unprocessed.items():
    num_entries = df.shape[0]
    num_na = df.query('item_name.isna()', engine='python').shape[0]
    num_duplicated = df[df.duplicated()].shape[0]
    total_entries += num_entries
    total_na += num_na
    total_duplicated += num_duplicated
    print(loc_id, num_duplicated)
print("Total entries: ", total_entries)
print("Total NA: ", total_na)
print("Total duplicated: ", total_duplicated)

# Verify the correspondance between location IDs in the reference data and the sales data

# Static data
locations_batch_1 = static_data_separated['locations_1']['location_id'].values # Identify the list of location ids for batch 1
locations_batch_2 = static_data_separated['locations_2']['location_id'].values # Identify the list of location ids for batch 2
locations_static_data = np.r_[locations_batch_1, locations_batch_2] # Concat the two batches (np.r_ is shorthand for concat)
locations_static_data.sort()
locations_df_static_data = pd.DataFrame(locations_static_data, columns=['location_id'])

# Sales data
location_filenames = os.listdir("data/0_data_excel/batch_1/orders_item_level") + os.listdir("data/0_data_excel/batch_2/orders_item_level") # Concatenate the filenames
locations_sales_data = [re.sub(r'\.xlsx$', '', filename) for filename in location_filenames] # Remove file endings
locations_df_sales_data = pd.DataFrame(locations_sales_data, columns=['location_id'])

# Match by visual inspection (hence the column concat)
location_id_matching = pd.concat([locations_df_static_data, locations_df_sales_data], axis=1)


# View data columns

# Initialize a list of rows of column names, where each row corresponds to the columns of a data frame
data_columns = []
for name, df in static_data_separated.items():

    # Add row of names to check that column matches
    data_columns.append([name] + df.columns.tolist())

# Same for sales data
for name, df in restaurant_data.items():

    # Add row of names to check that column matches
    data_columns.append([name] + df.columns.tolist())

# Make dataframe for viewing all the column names in all the data frames
data_columns_df = pd.DataFrame(data_columns)


# Compare batch 1's items_tagged v1 shape versus v2 shape

# Tagged items data batch 1 v1 
print(items_tagged_1.shape)

# Tagged items data batch 1 v2
print(items_tagged_v2_1.shape)

# Tagged items data batch 2
print(items_tagged_2.shape)



# Compare batch 1's items_tagged v1 values versus v2 values

# Match the part of version 2 that corresponds with verison 1
items_tagged_v2_1_modified = items_tagged_v2_1.iloc[:items_tagged_1.shape[0], :items_tagged_1.shape[1]]

# Standardize formatting
items_tagged_1_modified = (items_tagged_1.assign(is_plant_based = lambda x: x.is_plant_based.astype(str).str.lower().str.replace('.', ''))
                           .fillna(0)
                           .copy()) # NaNs are always treated as unequal, so temporarily fill them
items_tagged_v2_1_modified = (items_tagged_v2_1_modified.assign(is_plant_based = lambda x: x.is_plant_based.astype(str).str.lower().str.replace('.', ''))
                              .fillna(0)
                              .copy())

# Check if there are differences
changes = (items_tagged_1_modified != items_tagged_v2_1_modified).apply(lambda r: r.any(), axis=1)
changes.sum()

# There are no differences so v1 has strictly less data and is useless, and we won't carry it forward in the data processing